Mise en place et importation des data

In [47]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

data = pd.read_csv('SAE/2022/SYGEN_2022r.csv', encoding='latin1', sep=';')
data.head()

,FI,FI_EJ,AN,LIT_MED,LIT_CHI,LIT_OBS,LIT_MCO,SEJHC_MED,SEJHC_CHI,SEJHC_OBS,...,ACT_SCAN,NB_IRM,ACT_IRM,NB_CAM,ACT_CAM,NB_TOMO,ACT_TOMO,SAL_INTERV,POST_REVEIL,BOR
0,010000024,010780054,2022,270.0,68.0,47.0,385.0,14972.0,7260.0,3474.0,...,1.0,3.0,0.0,NaN,NaN,NaN,NaN,13.0,19.0,SYGEN
1,010000032,010780062,2022,66.0,17.0,8.0,91.0,4020.0,970.0,501.0,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,4.0,6.0,SYGEN
2,010000065,010780096,2022,59.0,NaN,NaN,59.0,1669.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SYGEN
3,010000081,010780112,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SYGEN
4,010000099,010780120,2022,10.0,NaN,NaN,10.0,170.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SYGEN


On s'intéresse maintenant aux MCO et aux ETP correspondants

In [12]:
séjours = data[['FI','SEJHC_MCO','SEJHP_MCO']]
variables = data[['FI','ETPSAL_SPEMED', 'ETPSAL_SPECHI', 'ETP_INFAVECSPE', 'ETP_INFSANSSPE', 'ETP_AID', 'ETP_ASH']]
séjours.head()

,FI,SEJHC_MCO,SEJHP_MCO
0,010000024,25706.0,845.0
1,010000032,5491.0,1703.0
2,010000065,1669.0,0.0
3,010000081,NaN,NaN
4,010000099,170.0,0.0


In [13]:
variables.head()

,FI,ETPSAL_SPEMED,ETPSAL_SPECHI,ETP_INFAVECSPE,ETP_INFSANSSPE,ETP_AID,ETP_ASH
0,010000024,115.07,27.59,84.64,406.03,340.65,98.60
1,010000032,20.80,7.87,6.86,87.64,76.74,41.90
2,010000065,14.51,NaN,NaN,59.91,88.58,7.42
3,010000081,1.76,NaN,NaN,7.43,16.66,22.75
4,010000099,2.20,NaN,NaN,7.80,14.48,6.21


Première estimation

In [40]:
Y = séjours['SEJHC_MCO']
X = variables[['ETPSAL_SPEMED', 'ETPSAL_SPECHI', 'ETP_INFAVECSPE', 'ETP_INFSANSSPE', 'ETP_AID', 'ETP_ASH']]

In [41]:
# Remplace tous les NaN par 0 dans X pour le moment. 
X = X.dropna()
Y = Y.loc[X.index]

In [42]:
X = sm.add_constant(X)

model = sm.OLS(Y, X)
results_robust = model.fit(cov_type='HC1')

print(results_robust.summary())

ValueError: r_matrix performs f_test for using dimensions that are asymptotically non-normal

In [44]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# --- 1. SÉLECTION ET CONVERSION ---
# On reprend vos variables (X_raw) et votre cible (Y_raw)
cols_X = ['ETPSAL_SPEMED', 'ETPSAL_SPECHI', 'ETP_INFAVECSPE', 
          'ETP_INFSANSSPE', 'ETP_AID', 'ETP_ASH']

# Force la conversion en numérique (transforme les erreurs/textes en NaN)
# C'est CRUCIAL car la SAE contient souvent des espaces ou des caractères invisibles
X_clean = X[cols_X].apply(pd.to_numeric, errors='coerce')
Y_clean = pd.to_numeric(Y, errors='coerce')

# --- 2. NETTOYAGE DES VALEURS MANQUANTES ---
# On combine pour supprimer les lignes qui ont des NaN dans X OU Y
combined = pd.concat([Y_clean, X_clean], axis=1).dropna()

# On sépare à nouveau
Y_final = combined.iloc[:, 0]
X_final = combined.iloc[:, 1:]

# --- 3. MODÉLISATION ---
if X_final.shape[1] == 0:
    print("ERREUR CRITIQUE : Toutes les variables explicatives ont une variance nulle.")
else:
    # Ajout de la constante sur le dataset PROPRE
    X_final = sm.add_constant(X_final)

    # Modèle
    model = sm.OLS(Y_final, X_final)
    
    # On retente avec HC1 maintenant que la matrice est propre
    try:
        results = model.fit(cov_type='HC1')
        print(results.summary())
    except Exception as e:
        print(f"HC1 échoue encore (échantillon trop petit ?) : {e}")
        print("Tentative OLS standard...")
        print(model.fit().summary())

                            OLS Regression Results                            
Dep. Variable:              SEJHC_MCO   R-squared:                       0.893
Model:                            OLS   Adj. R-squared:                  0.892
Method:                 Least Squares   F-statistic:                     269.9
Date:                Thu, 05 Feb 2026   Prob (F-statistic):          3.10e-164
Time:                        14:03:24   Log-Likelihood:                -5706.4
No. Observations:                 587   AIC:                         1.143e+04
Df Residuals:                     580   BIC:                         1.146e+04
Df Model:                           6                                         
Covariance Type:                  HC1                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const           1323.6197    299.841      4.

PArtie chat pour tout MCO

In [48]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# --- 1. CONFIGURATION DES VARIABLES ---
# Vérifiez que ces noms correspondent EXACTEMENT à vos colonnes
VARS_CONFIG = {
    "MÉDECINE": {
        "Y": "SEJHC_MED",  # Nombre de séjours en Médecine
        "X_SPECIFIQUE": ["ETPSAL_SPEMED"]
    },
    "CHIRURGIE": {
        "Y": "SEJHC_CHI",  # Nombre de séjours en Chirurgie
        "X_SPECIFIQUE": ["ETPSAL_SPECHI"]
    },
    "OBSTÉTRIQUE": {
        "Y": "SEJHC_OBS",  # Nombre de séjours en Obstétrique
        "X_SPECIFIQUE": ["ETP_SAG", "ETPSAL_DT_GYNOBS"]
    }
}

# Personnel commun à tous les services (Socle commun)
COMMON_STAFF = ['ETP_INFAVECSPE', 'ETP_INFSANSSPE', 'ETP_AID', 'ETP_ASH']

# --- 2. MOTEUR DE RÉGRESSION ROBUSTE ---
def run_model(df_source, name, y_col, x_cols):
    print(f"\n{'#'*30}")
    print(f"### MODÈLE : {name}")
    print(f"### Cible (Y) : {y_col}")
    print(f"{'#'*30}")

    # A. Vérification d'existence
    all_cols = [y_col] + x_cols
    missing = [c for c in all_cols if c not in df_source.columns]
    if missing:
        print(f"⛔ ARRET : Colonnes introuvables dans 'data' : {missing}")
        return

    # B. Création du sous-ensemble et conversion numérique (Blindage)
    df_work = df_source[all_cols].copy()
    
    # On force tout en numérique (les "nc", espaces, erreurs deviennent NaN)
    for col in df_work.columns:
        df_work[col] = pd.to_numeric(df_work[col], errors='coerce')

    # C. Nettoyage des lignes (l'alignement se fait ici)
    n_init = len(df_work)
    df_work = df_work.dropna()
    n_final = len(df_work)
    
    print(f"-> Nettoyage : {n_init} lignes au départ -> {n_final} lignes propres.")
    if n_final < 10:
        print("⛔ ARRET : Pas assez de données (<10 observations).")
        return

    # D. Séparation et Nettoyage de la collinéarité
    Y = df_work[y_col]
    X = df_work[x_cols]
    
    # Suppression des colonnes à variance nulle (qui font planter le modèle)
    # Ex: Si on modélise l'obstétrique mais qu'on a que des cliniques sans mater, 
    # ETP_SAGEFEMME sera à 0 partout. Il faut l'enlever dynamiquement.
    vars_stables = X.columns[X.std() > 0]
    vars_removed = set(x_cols) - set(vars_stables)
    
    if vars_removed:
        print(f"⚠️ Variables exclues (constantes/nulles) : {vars_removed}")
    
    X = X[vars_stables]
    
    if X.empty:
        print("⛔ ARRET : Plus aucune variable explicative valide.")
        return

    # E. Modélisation
    X = sm.add_constant(X)
    model = sm.OLS(Y, X)

    try:
        # Essai Robuste (HC1)
        results = model.fit(cov_type='HC1')
        print(f"✅ SUCCÈS (Robust HC1) | R² : {results.rsquared:.3f}")
        # Affichage propre des coefficients significatifs
        print(results.summary().tables[1])
        
    except Exception as e:
        print(f"⚠️ Échec HC1 ({e})... Tentative OLS standard")
        # Fallback OLS classique
        results = model.fit()
        print(f"✅ SUCCÈS (Standard OLS) | R² : {results.rsquared:.3f}")
        print(results.summary().tables[1])

# --- 3. EXÉCUTION ---
# On boucle sur le dictionnaire
for sector, config in VARS_CONFIG.items():
    # Construction de la liste X complète pour ce secteur
    x_sector = config["X_SPECIFIQUE"] + COMMON_STAFF
    
    # Lancement
    run_model(data, sector, config["Y"], x_sector)


##############################
### MODÈLE : MÉDECINE
### Cible (Y) : SEJHC_MED
##############################
-> Nettoyage : 3988 lignes au départ -> 848 lignes propres.
✅ SUCCÈS (Robust HC1) | R² : 0.845
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const            705.8046    186.434      3.786      0.000     340.401    1071.208
ETPSAL_SPEMED     23.7810      8.517      2.792      0.005       7.087      40.475
ETP_INFAVECSPE    19.7811      6.623      2.987      0.003       6.800      32.762
ETP_INFSANSSPE    11.4361      3.126      3.659      0.000       5.310      17.563
ETP_AID            5.0791      2.841      1.788      0.074      -0.489      10.647
ETP_ASH            4.6104      4.560      1.011      0.312      -4.328      13.549

##############################
### MODÈLE : CHIRURGIE
### Cible (Y) : SEJHC_CHI
##############################
-> Nettoyage : 398